In [3]:
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

import joblib

In [4]:
df1 = pd.read_csv("../data/raw/cleaned_reviews-sentiments.csv")
df2 = pd.read_csv("../data/raw/IMDB Dataset-sentiments.csv")
df3 = pd.read_csv("../data/raw/sentiment-analysis.csv")
df4 = pd.read_csv("../data/raw/Tweets-sentiments.csv")

print(df1.columns)
print(df2.columns)
print(df3.columns)
print(df4.columns)

Index(['sentiments', 'cleaned_review', 'cleaned_review_length',
       'review_score'],
      dtype='str')
Index(['review', 'sentiment'], dtype='str')
Index(['Text, Sentiment, Source, Date/Time, User ID, Location, Confidence Score'], dtype='str')
Index(['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence',
       'negativereason', 'negativereason_confidence', 'airline',
       'airline_sentiment_gold', 'name', 'negativereason_gold',
       'retweet_count', 'text', 'tweet_coord', 'tweet_created',
       'tweet_location', 'user_timezone'],
      dtype='str')


In [ ]:
import pandas as pd

#  Charger les CSV avec le bon séparateur et gestion des guillemets
df1 = pd.read_csv("../data/raw/cleaned_reviews-sentiments.csv", sep=",", quotechar='"', engine='python')
df2 = pd.read_csv("../data/raw/IMDB Dataset-sentiments.csv", sep=",", quotechar='"', engine='python')

# Pour df3, tout est dans une seule colonne, on la charge brute
df3 = pd.read_csv("../data/raw/sentiment-analysis.csv", header=None, names=['raw'], engine='python')

df4 = pd.read_csv("../data/raw/Tweets-sentiments.csv", sep=",", quotechar='"', engine='python')

#  Prétraiter df3 pour extraire text et sentiment
# On split sur la première virgule (text, sentiment)
df3[['text', 'sentiment']] = df3['raw'].str.split(',', n=1, expand=True)
# Nettoyer les guillemets et espaces
df3['text'] = df3['text'].str.strip().str.replace('"', '')
df3['sentiment'] = df3['sentiment'].str.strip().str.replace('"', '')

#  Renommer les autres colonnes pour uniformiser
df1 = df1.rename(columns={'cleaned_review': 'text', 'sentiments': 'sentiment'})
df2 = df2.rename(columns={'review': 'text', 'sentiment': 'sentiment'})
df4 = df4.rename(columns={'text': 'text', 'airline_sentiment': 'sentiment'})

#  Sélectionner uniquement les colonnes texte + sentiment
df1 = df1[['text', 'sentiment']]
df2 = df2[['text', 'sentiment']]
df3 = df3[['text', 'sentiment']]
df4 = df4[['text', 'sentiment']]

#  Fusionner les datasets
data = pd.concat([df1, df2, df3, df4], ignore_index=True)

#  Vérifier le résultat
print("Taille du dataset fusionné :", data.shape)
print("Colonnes :", list(data.columns))
print(data.head())

Taille du dataset fusionné : (82077, 2)
Colonnes : ['text', 'sentiment']
                                                text sentiment
0  i wish would have gotten one earlier love it a...  positive
1  i ve learned this lesson again open the packag...   neutral
2          it is so slow and lags find better option   neutral
3  roller ball stopped working within months of m...   neutral
4  i like the color and size but it few days out ...   neutral


In [6]:
# Nettoyer les labels
data['sentiment'] = data['sentiment'].astype(str)  # assure que tout est string
data['sentiment'] = data['sentiment'].str.strip()  # enlever espaces
data['sentiment'] = data['sentiment'].str.split(',').str[0]  # prendre seulement le premier élément avant la virgule
data['sentiment'] = data['sentiment'].str.lower()  # mettre en minuscules

# Vérifier les labels uniques
print(data['sentiment'].value_counts())

sentiment
positive     36919
negative     35755
neutral       9402
sentiment        1
Name: count, dtype: int64


In [ ]:
# Cellule 3 : Nettoyage du texte

import pandas as pd
import re

#  Vérifier les colonnes du dataset
print("Colonnes avant nettoyage :", list(data.columns))

#  Fonction de nettoyage du texte
def clean_text(text):
    text = str(text).lower()                     # tout en minuscules
    text = re.sub(r"http\S+", "", text)          # enlever les URLs
    text = re.sub(r"[^a-z\s]", "", text)        # enlever chiffres et ponctuation
    text = re.sub(r"\s+", " ", text).strip()    # enlever espaces multiples
    return text

#  Appliquer le nettoyage
data["clean_text"] = data["text"].apply(clean_text)

#  Supprimer les lignes vides après nettoyage
data = data[data["clean_text"] != ""]

#  Normaliser les labels sentiment (prendre seulement positive/negative/neutral)
data['sentiment'] = data['sentiment'].astype(str).str.strip().str.split(',').str[0].str.lower()
valid_labels = ['positive', 'negative', 'neutral']
data = data[data['sentiment'].isin(valid_labels)]

#  Vérifier le dataset final
print("Colonnes après nettoyage :", list(data.columns))
print("Taille du dataset après nettoyage :", data.shape)
print(data['sentiment'].value_counts())
print(data.head(10))

Colonnes avant nettoyage : ['text', 'sentiment']
Colonnes après nettoyage : ['text', 'sentiment', 'clean_text']
Taille du dataset après nettoyage : (82060, 3)
sentiment
positive    36919
negative    35755
neutral      9386
Name: count, dtype: int64
                                                text sentiment  \
0  i wish would have gotten one earlier love it a...  positive   
1  i ve learned this lesson again open the packag...   neutral   
2          it is so slow and lags find better option   neutral   
3  roller ball stopped working within months of m...   neutral   
4  i like the color and size but it few days out ...   neutral   
5  overall love this mouse the size weight clicki...  positive   
6                                 it stopped working   neutral   
7  my son uses school issued chromebook for schoo...  positive   
8  loved this cute little mouse but it broke afte...  negative   
9  should ve spent the money to get quality produ...  negative   

                        

In [ ]:
# ---------------------------------
# Cellule 4 : Texte → vecteurs TF-IDF
# ---------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer

#  Initialiser le TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

#  Transformer le texte nettoyé en vecteurs numériques
X = vectorizer.fit_transform(data["clean_text"])

#  Variable cible (sentiment)
y = data["sentiment"]

#  Vérifier la dimension
print("Shape de X :", X.shape)  # (nombre d'avis, nombre de features)
print("Nombre de labels :", y.shape)

#  Aperçu des sentiments uniques
print("Sentiments uniques :", y.unique())

Shape de X : (82060, 5000)
Nombre de labels : (82060,)
Sentiments uniques : ['positive' 'neutral' 'negative']


In [ ]:
# ---------------------------------
# Cellule 4 : Texte → vecteurs TF-IDF
# ---------------------------------
#-- correction des espaces ici : ['positive' 'neutral' 'negative']
from sklearn.feature_extraction.text import TfidfVectorizer

# nettoyer les espaces dans les labels
data["sentiment"] = data["sentiment"].str.strip()

#  Initialiser le vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

#  Transformer le texte nettoyé en vecteurs numériques
X = vectorizer.fit_transform(data["clean_text"])

#  Variable cible
y = data["sentiment"]

#  Vérifications
print("Shape de X :", X.shape)
print("Nombre de labels :", y.shape)
print("Sentiments uniques :", y.unique())

Shape de X : (82060, 5000)
Nombre de labels : (82060,)
Sentiments uniques : ['positive' 'neutral' 'negative']


In [ ]:
# -------------------------------
# Cellule 5 : Split Train/Test            Train :  pour entraîner le modèle
# -------------------------------         Test  :  pour vérifier la performance du modèle 



from sklearn.model_selection import train_test_split

# 1- Séparer les données
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,     # 20% pour le test
    random_state=42    # pour reproduire les mêmes résultats
)

# 2️- Vérifier les dimensions
print("Train size :", X_train.shape)
print("Test size :", X_test.shape)
print("Train labels :", y_train.shape)
print("Test labels :", y_test.shape)


# Explication pour affichage  :

# 76 avis pour entraîner le modèle
# 20 avis pour tester le modèle
# 271 features (mots TF-IDF)

Train size : (65648, 5000)
Test size : (16412, 5000)
Train labels : (65648,)
Test labels : (16412,)


In [ ]:
# Cellule 6 : Entraîner le modèle

from sklearn.linear_model import LogisticRegression

#  Créer le modèle
model = LogisticRegression(max_iter=1000)

#  Entraîner le modèle avec les données train
model.fit(X_train, y_train)

#  Message de confirmation
print("Modèle entraîné avec succès")

Modèle entraîné avec succès


In [ ]:
# -------------------------------
# Cellule 7 : Tester(ou evaluer) le modèle
# -------------------------------

from sklearn.metrics import accuracy_score, classification_report

#  Faire les prédictions sur les données de test
predictions = model.predict(X_test)

#  Calculer l'accuracy globale
accuracy = accuracy_score(y_test, predictions)
print("Accuracy :", accuracy)

#  Afficher le rapport détaillé
print(classification_report(y_test, predictions))


#faire moi attention je peux avoir un Overfitting: possible que le modèle apprenne trop facilement les patterns,ce n’est pas forcément un modèle parfait car dataset est très petit

Accuracy : 0.8495003655861565
              precision    recall  f1-score   support

    negative       0.86      0.87      0.86      7191
     neutral       0.73      0.67      0.70      1875
    positive       0.87      0.88      0.87      7346

    accuracy                           0.85     16412
   macro avg       0.82      0.81      0.81     16412
weighted avg       0.85      0.85      0.85     16412



In [ ]:
# Cellule 7bis : Cross Validation


from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Initialiser le modèle
cv_model = LogisticRegression(max_iter=1000)

# stratified k-fold 
# garde la même proportion de classes dans chaque fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross validation
scores = cross_val_score(cv_model, X, y, cv=cv)

print("Scores de chaque fold :", scores)
print("Accuracy moyenne :", scores.mean())

Scores de chaque fold : [0.85260785 0.84700219 0.85047526 0.84590544 0.85010968]
Accuracy moyenne : 0.8492200828661953


In [ ]:
# Cellule 8 : Sauvegarder le modèle et le vectorizer


import os
import joblib

#  Dossier où on veut sauvegarder
model_dir = "../backend/app/ml/models"

#  Créer le dossier s'il n'existe pas
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

#  Chemins complets des fichiers
model_path = os.path.join(model_dir, "sentiment_model.pkl")
vectorizer_path = os.path.join(model_dir, "vectorizer.pkl")

#  Sauvegarder le modèle entraîné
joblib.dump(model, model_path)
print(f"✅ Modèle sauvegardé : {model_path}")

#  Sauvegarder le vectorizer
joblib.dump(vectorizer, vectorizer_path)
print(f"✅ Vectorizer sauvegardé : {vectorizer_path}")

✅ Modèle sauvegardé : ../backend/app/ml/models\sentiment_model.pkl
✅ Vectorizer sauvegardé : ../backend/app/ml/models\vectorizer.pkl


In [ ]:
# -------------------------------
# Cellule 9 : Tester le modèle dans le notebook
# -------------------------------

import joblib

#  Chemins vers les fichiers sauvegardés
model_path = "../backend/app/ml/models/sentiment_model.pkl"
vectorizer_path = "../backend/app/ml/models/vectorizer.pkl"

#  Recharger le modèle et le vectorizer
model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)

#  Nouvelles phrases à analyser
test_texts = [
    "The service was amazing",
    "I am very disappointed with the product",
    "It was okay, nothing special"
]

#  Transformer le texte en vecteurs
X_test_vector = vectorizer.transform(test_texts)

#  Faire la prédiction
predictions = model.predict(X_test_vector)

#  Afficher les résultats
for text, pred in zip(test_texts, predictions):
    print(f"Texte : {text} --> Sentiment prédit : {pred}")

Texte : The service was amazing --> Sentiment prédit : positive
Texte : I am very disappointed with the product --> Sentiment prédit : negative
Texte : It was okay, nothing special --> Sentiment prédit : neutral


In [16]:
def predict_sentiment(text):
    vector = vectorizer.transform([text])
    return model.predict(vector)[0]

print(predict_sentiment("I love this product!"))
print(predict_sentiment("Worst experience ever"))
print(predict_sentiment("The product arrived yesterday and I used it today."))

positive
negative
neutral


In [17]:
def predict_sentiment(text):
    vector = vectorizer.transform([text])
    return model.predict(vector)[0]

print(predict_sentiment("The customer service was outstanding!"))                  # positive
print(predict_sentiment("I am never buying from this company again."))             # negative   -- probleme ici le model a donner neutre 
print(predict_sentiment("My package arrived on time and everything works."))       # positive   -- probleme ici le model a donner neutre 
print(predict_sentiment("Absolutely thrilled with my purchase!"))                  # positive
print(predict_sentiment("The product broke after one use, very disappointed."))    # negative
print(predict_sentiment("Just received my order, nothing special."))               # neutral
print(predict_sentiment("Fast delivery and excellent quality."))                   # positive
print(predict_sentiment("Emails went unanswered, terrible support."))              # negative
print(predict_sentiment("The website was easy to navigate."))                      # positive
print(predict_sentiment("Highly recommend this company to everyone!"))             # positive

positive
neutral
neutral
positive
negative
neutral
positive
negative
positive
positive


In [18]:
# Exemple pour vérifier
print(type(data))  # doit être <class 'pandas.core.frame.DataFrame'>
print(data.columns)

<class 'pandas.DataFrame'>
Index(['text', 'sentiment', 'clean_text'], dtype='str')


In [19]:
# Supposons que ton DataFrame NLP s'appelle `data`
# Transformer sentiment en score numérique
data['sentiment_score'] = data['sentiment'].map({
    'positive': 1,
    'neutral': 0,
    'negative': -1
})

# Sauvegarder dans processed/
data.to_csv("../data/processed/nlp_sentiment_scores.csv", index=False)

print("NLP data avec sentiment_score sauvegardé !")

NLP data avec sentiment_score sauvegardé !


In [ ]:
list(data.columns)


['text', 'sentiment', 'clean_text', 'sentiment_score']